# Step 8: Scale Your Prototype with Large-Scale Data

Capstone: Traffic Speed Prediction Using California PeMS Data

This notebook demonstrates a scalable prototype using chunked processing and incremental learning.

## Repository and Dataset

Repository: https://github.com/rarra21/Springboard

Dataset file: https://github.com/rarra21/Springboard/blob/main/sample_pems_data_small.csv

Raw URL: `https://raw.githubusercontent.com/rarra21/Springboard/main/sample_pems_data_small.csv`

In [1]:
import os, time, joblib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
LOCAL_PATH = 'sample_pems_data_small.csv'
GITHUB_PATH = 'https://raw.githubusercontent.com/rarra21/Springboard/main/sample_pems_data_small.csv'
DATA_PATH = LOCAL_PATH if os.path.exists(LOCAL_PATH) else GITHUB_PATH
FIG_DIR = Path('figures'); RESULT_DIR = Path('results'); MODEL_DIR = Path('models')
FIG_DIR.mkdir(exist_ok=True); RESULT_DIR.mkdir(exist_ok=True); MODEL_DIR.mkdir(exist_ok=True)
print('Data path:', DATA_PATH)

Data path: sample_pems_data_small.csv


In [2]:
df = pd.read_csv(DATA_PATH)
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(['timestamp','station_id']).reset_index(drop=True)
print('Shape:', df.shape)
df.head()

Shape: (160, 9)


## Why scaling is needed

The Step 7 prototype used small in-memory data. A real PeMS deployment may contain millions of readings across many stations, so the pipeline should avoid requiring the full dataset in memory.

In [3]:
features = ['station_id','flow','occupancy','speed','hour','dayofweek']
target = 'speed_60min_ahead'
df = df.dropna(subset=features + [target]).reset_index(drop=True)
split = int(len(df) * 0.8)
train_df = df.iloc[:split].copy()
test_df = df.iloc[split:].copy()
X_train = train_df[features].astype(float).values
y_train = train_df[target].astype(float).values
X_test = test_df[features].astype(float).values
y_test = test_df[target].astype(float).values
print('Train rows:', len(train_df), 'Test rows:', len(test_df))

Train rows: 128 Test rows: 32


## Incremental scaling approach

This prototype uses `StandardScaler.partial_fit` and `SGDRegressor.partial_fit`. This allows the pipeline to train in chunks instead of loading all data at once.

In [4]:
chunk_size = 32
scaler = StandardScaler()
for start in range(0, len(X_train), chunk_size):
    scaler.partial_fit(X_train[start:start+chunk_size])

model = SGDRegressor(loss='squared_error', penalty='l2', alpha=0.0001,
                     max_iter=1, tol=None, random_state=RANDOM_STATE,
                     learning_rate='invscaling', eta0=0.001)
train_mae_history = []
start_time = time.time()
for epoch in range(25):
    for start in range(0, len(X_train), chunk_size):
        Xb = scaler.transform(X_train[start:start+chunk_size])
        yb = y_train[start:start+chunk_size]
        model.partial_fit(Xb, yb)
    train_pred = model.predict(scaler.transform(X_train))
    train_mae_history.append(mean_absolute_error(y_train, train_pred))
train_time = time.time() - start_time
print('Training completed in seconds:', round(train_time, 4))

Training completed in seconds: 0.0386


In [5]:
pred_start = time.time()
pred = model.predict(scaler.transform(X_test))
prediction_time = time.time() - pred_start
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
corr = np.corrcoef(y_test, pred)[0,1]
metrics = pd.DataFrame([{
    'model':'Incremental SGDRegressor',
    'MAE':mae,
    'RMSE':rmse,
    'R2':r2,
    'Correlation':corr,
    'Training Time Sec':train_time,
    'Prediction Time Sec':prediction_time
}])
metrics

In [6]:
joblib.dump({'scaler': scaler, 'model': model, 'features': features}, MODEL_DIR/'step8_incremental_sgd_model.joblib')
metrics.to_csv(RESULT_DIR/'step8_scaling_results.csv', index=False)
pd.DataFrame({'actual':y_test, 'predicted':pred}).to_csv(RESULT_DIR/'step8_predictions.csv', index=False)
pd.DataFrame({'epoch':range(1, len(train_mae_history)+1), 'train_mae':train_mae_history}).to_csv(RESULT_DIR/'step8_training_curve.csv', index=False)
print('Saved model, metrics, and predictions.')

Saved model, metrics, and predictions.


In [7]:
plt.figure(figsize=(8,4.5))
plt.plot(range(1, len(train_mae_history)+1), train_mae_history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Training MAE')
plt.title('Incremental Training Curve')
plt.tight_layout()
plt.savefig(FIG_DIR/'incremental_training_curve.png', dpi=160)
plt.show()

In [8]:
plt.figure(figsize=(8,4.5))
plt.plot(range(len(y_test)), y_test, marker='o', label='Actual')
plt.plot(range(len(pred)), pred, marker='x', label='Predicted')
plt.xlabel('Test Observation')
plt.ylabel('Speed 60 min ahead')
plt.title('Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR/'actual_vs_predicted_step8.png', dpi=160)
plt.show()

In [9]:
row_sizes = [len(df), len(df)*5, len(df)*10, len(df)*25, len(df)*50]
timing_rows=[]
for n in row_sizes:
    big = pd.concat([df]*int(np.ceil(n/len(df))), ignore_index=True).iloc[:n]
    t0=time.time()
    xb = scaler.transform(big[features].astype(float).values)
    _ = model.predict(xb)
    timing_rows.append({'rows': n, 'prediction_pipeline_time_sec': time.time()-t0})
scale_df = pd.DataFrame(timing_rows)
scale_df.to_csv(RESULT_DIR/'step8_scaling_timing_demo.csv', index=False)
scale_df

## Optional SparkML production direction

For complete PeMS-scale data, Spark can load partitioned data and distribute preprocessing/training across a cluster. The following code is a template for future use.

In [ ]:
# Optional SparkML template for future distributed execution.
# from pyspark.sql import SparkSession
# from pyspark.ml.feature import VectorAssembler, StandardScaler
# from pyspark.ml.regression import GBTRegressor
# from pyspark.ml import Pipeline
# spark = SparkSession.builder.appName('PeMS_Scaled_Traffic_Prediction').getOrCreate()
# sdf = spark.read.csv(GITHUB_PATH, header=True, inferSchema=True)
# assembler = VectorAssembler(inputCols=features, outputCol='raw_features')
# scaler = StandardScaler(inputCol='raw_features', outputCol='features')
# gbt = GBTRegressor(featuresCol='features', labelCol=target)
# pipeline = Pipeline(stages=[assembler, scaler, gbt])
# model = pipeline.fit(sdf)
# predictions = model.transform(sdf)

## Conclusion

The scalable prototype demonstrates chunked processing and incremental learning. This reduces memory pressure and creates a clearer path to production-scale PeMS forecasting. SparkML is recommended for the next phase if the project moves to complete PeMS data or cloud infrastructure.